# 04 -- Create Staff Accounts

Staff-migration variant of `04_Create_Accounts.ipynb`. Same account-cleaning
/ contact-attaching / idempotent-creation logic, with three differences:

1. **Scoped to staff accounts that actually have an active subscription.**
   We pull *every* staff account, separately pull *every active staff
   subscription*, then only create the accounts that show up in both -- an
   account with no active subscription doesn't get created at all. (Both
   MySQL queries below are placeholders -- fill in the real `WHERE` clauses
   for "staff account" and "active subscription" before running.)
2. **No Williams-style bucket accounts.** Every staff account is a real,
   its-own OneBill account -- there's no `managed_by_williams` /
   `williams_corporation` routing concept here.
3. **`accountNumber` is auto-generated by OneBill**, not chosen by us. We
   don't send one in the create payload (`build_account_payload(...,
   include_account_number=False)`); instead we capture whatever OneBill
   assigns from the create response (`extract_generated_account_number` in
   `onebill_common.py`) and save it as `GeneratedAccountNumber`.
   `05_Fetch_Staff_Subscriptions.ipynb` joins that back onto each
   subscription (via the original vBill `AccountCode`) to get the
   `TargetAccountNumber` the order actually gets attached to.

   **This response-parsing is unconfirmed against the real API** -- see the
   TODO on `GENERATED_ACCOUNT_NUMBER_KEYS` in `onebill_common.py`. Run this
   notebook against ONE staff account first, inspect the raw response, and
   adjust `extract_generated_account_number` if the real field name differs
   before doing a full run. As a safety net, the original vBill `AccountCode`
   is also stamped onto the account as an `accountAttribute` ("vBill Account
   Code") so the account is still identifiable even if the generated-number
   extraction needs fixing after the fact.

   Also note the **idempotency gap** this creates: a re-run that hits
   `"exists"` (account already created by an earlier run) has no way to
   recover that account's `GeneratedAccountNumber` from the "exists"
   response -- see the docstring on `migrate_staff_account` in
   `onebill_common.py`. Keep earlier `staff_account_results.csv` output
   around rather than assuming a second run will backfill it.

**This is written for a full bulk migration** (every qualifying staff
account) -- set `TEST_ROW_LIMIT = None` when you're ready.

**Idempotency** -- same three-way contract as `04_Create_Accounts.ipynb`:
`"created"` / `"exists"` (not a failure -- proceed as normal, but see the
note above) / `"failed"`.

## 1. Setup

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403
from sqlalchemy import create_engine
from concurrent.futures import ThreadPoolExecutor, as_completed

logger = get_logger("create_staff_accounts")
session = new_session(max_workers=MAX_WORKERS)

# Use this to limit rows while testing. Set to None once ready for a full run.
TEST_ROW_LIMIT = 5

# Appended to AccountName_Unique only (for disambiguating test-run account
# NAMES in OneBill's sandbox) -- NOT sent as accountNumber, since OneBill
# chooses that itself for this pipeline. Leave "" for a real/final run.
STAFF_TEST_SUFFIX = os.environ.get("STAFF_TEST_SUFFIX", "")


In [ ]:
# HARDCODED_TOKEN = "133cc573-dc96-4136-9834-92e1d3c8b55e"

# token_manager._token = HARDCODED_TOKEN
# token_manager._expires_at = datetime.now() + timedelta(hours=1)  # adjust to match the real token's actual TTL

## 2. Load contacts (from `01_Fetch_Contacts.ipynb`)

Unchanged -- the staff migration reuses `00`-`03` exactly as they are for the Williams migration; nothing staff-specific about contacts, products, or plan-code matching.

In [ ]:
df_contacts_raw = load_df("contacts_raw")
contacts_by_account = index_contacts_by_account(df_contacts_raw)
logger.info(
    f"Indexed {sum(len(v) for v in contacts_by_account.values()):,} contacts "
    f"across {len(contacts_by_account):,} accounts"
)


## 3. All staff accounts (MySQL)

**PLACEHOLDER** -- fill in the real condition that identifies a "staff"
account in `bi_datastore.billing_account` (e.g. `AccountType = 'Staff'`, or
whatever the real vBill flag/category turns out to be). Everything else
(name cleaning, Individual/Business classification, address fallbacks)
mirrors `04_Create_Accounts.ipynb`'s `ACCOUNT_QUERY` exactly -- no reason for
that part to differ.

In [ ]:
assert BI_DATASTORE_URL, "DB_USERNAME/DB_PASSWORD/DB_HOST not set in .env"
engine = create_engine(BI_DATASTORE_URL)

STAFF_ACCOUNT_QUERY = """
SELECT
    `AccountName` AS `AccountName_Original`
    ,TRIM(
        REPLACE(
            REPLACE(
                REPLACE(
                    REPLACE(
                        REPLACE(
                            REPLACE(
                                REPLACE(
                                    REPLACE(
                                        REPLACE(
                                            REPLACE(
                                                REPLACE(
                                                    REPLACE(
                                                        REPLACE(
                                                            REPLACE(
                                                                REPLACE(
                                                                    REPLACE(`AccountName`, '(BOND - DECLINED)', ''),
                                                                '(BOND)', ''),
                                                            '(ICMS)', ''),
                                                        '(Staff)', ''),
                                                    '(X)', ''),
                                                '(In Liquidation)', ''),
                                            'zz-', ''),
                                        '(DECLINED)', ''),
                                    '(Operator)', ''),
                                '(Liquidation )', ''),
                            '(LIQUIDATION)', ''),
                        '(Under Liquidation)', ''),
                    '[LIQUIDATION]', ''),
                '(Bad - Debt)', ''),
            '(Bad Debt)', ''),
        '(Bad-Debt)', ''),
    '(BOND - DECLINED)', '')
    ) AS `AccountName_Cleaned`
    ,`AccountCode`
    ,`CreatedDate`
    ,`ClosedDate`
    ,`AccountType`
    ,CASE
        WHEN `AccountType` IN ('Residential', 'Actrix Residential', 'Consumer', 'Standard Account', 'Internal-Use Account', 'Staff')
        AND `AccountName` NOT LIKE '%Ltd%'
        AND `AccountName` NOT LIKE '%Limited%'
        AND `AccountName` NOT LIKE '%Pty%'
        THEN '1001' -- Individual Customer
        ELSE '1002' -- Business Customer
    END AS `OneBill_AccountType`
    ,CASE
        WHEN `_temporary_crmonly_addr1` IS NULL OR `_temporary_crmonly_addr1` = '' THEN '1 Somewhere Place'
        ELSE `_temporary_crmonly_addr1`
    END AS `Address1`
    ,`_temporary_crmonly_addr2` AS `Address2`
    ,`_temporary_crmonly_suburb` AS `Suburb`
    ,CASE
        WHEN `_temporary_crmonly_city` IS NULL OR `_temporary_crmonly_city` = '' THEN 'Auckland'
        ELSE `_temporary_crmonly_city`
    END AS `City`
    ,CASE
        WHEN `_temporary_crmonly_postcode` IS NULL OR `_temporary_crmonly_postcode` = '' THEN '0001'
        ELSE `_temporary_crmonly_postcode`
    END AS `Postcode`
    ,`_temporary_crmonly_dob` AS `DateOfBirth`
FROM
    bi_datastore.billing_account
WHERE
    `_DataSource` = 'vBill'
AND
    -- TODO: replace with the real "this is a staff account" condition, e.g.:
    -- `AccountType` = 'Staff'
    1 = 0
ORDER BY
    `AccountCode` DESC
""".strip()

df_staff_accounts_all = pd.read_sql(STAFF_ACCOUNT_QUERY, con=engine)
logger.info(f"Loaded {len(df_staff_accounts_all):,} staff accounts from MySQL (before active-subscription filter)")
df_staff_accounts_all.head()


## 4. Active subscriptions for staff accounts (MySQL)

**PLACEHOLDER** -- same idea as `SUBSCRIPTION_QUERY` in
`05_Fetch_Subscriptions.ipynb` (active = `SubscriptionEndDate IS NULL`), but
scoped to staff accounts only. Saved to `migration_data/` immediately
(`staff_subscriptions_active_raw`) so `05_Fetch_Staff_Subscriptions.ipynb`
loads this exact snapshot instead of re-querying MySQL -- guarantees the
subscriptions used for order creation are the SAME ones that decided which
accounts got created below, even if the underlying table changes between
notebook runs.

In [ ]:
STAFF_ACTIVE_SUBSCRIPTIONS_QUERY = """
SELECT * FROM bi_datastore.billing_subscription
WHERE _DataSource = 'vBill'
AND SubscriptionEndDate IS NULL
AND AccountCode IN (
    -- TODO: same staff condition as STAFF_ACCOUNT_QUERY above, applied to
    -- billing_account, e.g.:
    -- SELECT AccountCode FROM bi_datastore.billing_account
    -- WHERE _DataSource = 'vBill' AND AccountType = 'Staff'
    SELECT AccountCode FROM bi_datastore.billing_account WHERE 1 = 0
)
""".strip()

df_staff_subscriptions_active = pd.read_sql(STAFF_ACTIVE_SUBSCRIPTIONS_QUERY, con=engine)
logger.info(f"Loaded {len(df_staff_subscriptions_active):,} active staff subscriptions from MySQL")

save_df("staff_subscriptions_active_raw", df_staff_subscriptions_active)
df_staff_subscriptions_active.head()


## 5. Only create accounts that have at least one active subscription

Semi-join on `AccountCode` -- any staff account with zero rows in the
active-subscriptions query above is dropped here and never sent to OneBill
at all.

In [ ]:
active_account_codes = set(df_staff_subscriptions_active["AccountCode"].dropna().astype(str))

df_staff_accounts_all["AccountCode"] = df_staff_accounts_all["AccountCode"].astype(str)
has_active_subscription = df_staff_accounts_all["AccountCode"].isin(active_account_codes)

df_mysql_accounts = df_staff_accounts_all[has_active_subscription].copy()

logger.info(
    f"{len(df_staff_accounts_all):,} staff accounts total -> "
    f"{len(df_mysql_accounts):,} have at least one active subscription and will be created "
    f"({(~has_active_subscription).sum():,} skipped, no active subscription)"
)

# Unique display name -- no AccountCode_Batch here (nothing is appended to the
# real vBill AccountCode), since OneBill chooses the accountNumber itself for
# this pipeline. STAFF_TEST_SUFFIX only affects the NAME, only for test runs.
df_mysql_accounts["AccountName_Unique"] = (
    df_mysql_accounts["AccountName_Cleaned"] + " (" + df_mysql_accounts["AccountCode"] + STAFF_TEST_SUFFIX + ")"
)

if TEST_ROW_LIMIT is not None:
    df_mysql_accounts = df_mysql_accounts.head(TEST_ROW_LIMIT)  # Testing limiter -- remove/raise for a full run.
    logger.info(f"TEST_ROW_LIMIT active -- trimmed to {len(df_mysql_accounts):,} rows")

save_df("staff_accounts", df_mysql_accounts)
df_mysql_accounts.head()


## 6. Per-account worker

`migrate_staff_account` (in `onebill_common.py`) does the actual create call -- no accountNumber sent, generated number captured from the response. See its docstring for the idempotency caveat on `"exists"` hits.

In [ ]:
def create_all_staff_accounts(df: pd.DataFrame, max_workers: int = MAX_WORKERS) -> pd.DataFrame:
    rows = df.to_dict("records")
    total = len(rows)
    results = []

    logger.info(f"Creating {total:,} staff accounts with {max_workers} workers...")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(migrate_staff_account, row, session, contacts_by_account): row["AccountCode"]
            for row in rows
        }
        for i, future in enumerate(as_completed(futures), start=1):
            results.append(future.result())
            if i % 50 == 0 or i == total:
                ok = sum(1 for r in results if r["status"] == "created")
                logger.info(f"Progress: {i}/{total} -- {ok} created so far")

    return pd.DataFrame(results)


df_account_results = create_all_staff_accounts(df_mysql_accounts)
df_account_results.head(20)


## 7. Save + failure / missing-number summary

In [ ]:
save_df("staff_account_results", df_account_results)

failures = df_account_results[df_account_results["status"] == "failed"]
print(f"{len(failures):,} / {len(df_account_results):,} accounts failed to create")
failures.head(20)


In [ ]:
# Created OK, but no GeneratedAccountNumber came back -- these can't be routed
# to in 05_Fetch_Staff_Subscriptions.ipynb until extract_generated_account_number
# is fixed to match the real response shape (see the notebook intro).
created_no_number = df_account_results[
    (df_account_results["status"] == "created") & df_account_results["GeneratedAccountNumber"].isna()
]
print(f"{len(created_no_number):,} accounts created but no accountNumber could be extracted from the response")
created_no_number.head(20)
